# 3-Way TCRE Comparison: Felt vs Gel vs Paste

This notebook performs a comprehensive comparison between three types of
tripolar concentric ring electrodes (TCREs):

- **Felt TCRE** — all available recordings in `data/` with 11-channel layout
- **Gel TCRE** — from the Gel TCRE project recordings (7 channels)
- **Paste TCRE** — present in **both** recording setups as a bridge reference

Both datasets share: 1000 Hz sampling, 0.1 µV/bit resolution, BrainVision format,
checkerboard VEP + eyes open/closed paradigm.

> **Note on 9-channel subjects (Gab, LS1, TU):** these are early-prototype recordings
> with an unknown channel layout. They are automatically skipped until a
> `SHORT_FELT_9CH_CONFIG` is defined in `eeg_analysis.py`.

In [1]:
import sys, os, importlib
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt

# Ensure src/ is on the path
SRC_DIR = os.path.dirname(os.path.abspath('__file__'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import eeg_analysis, comparison_analysis
importlib.reload(eeg_analysis)
importlib.reload(comparison_analysis)

from eeg_analysis import (
    FELT_TCRE_CONFIG, GEL_TCRE_CONFIG,
    discover_subjects, load_subject, analyze_subject,
    plot_subject_summary, plot_subject_full,
)
from comparison_analysis import (
    load_all_subjects, print_qc_report,
    extract_type_metrics, extract_psd_by_type,
    extract_open_closed_psd_by_type, compare_electrode_types,
    plot_three_way_comparison, TYPE_COLORS, TYPE_DISPLAY,
)

## 1. Configuration

In [2]:
REPO_ROOT = os.path.abspath(os.path.join(SRC_DIR, '..'))

FELT_DATA_DIR = os.path.join(REPO_ROOT, 'data')
GEL_DATA_DIR = os.path.join(REPO_ROOT, 'Gel TCRE')
OUTPUT_DIR = os.path.join(REPO_ROOT, 'output', 'three_way_comparison')

# ── Subject selection ────────────────────────────────────────────────────
# FELT_SUBJECTS: None = try all discovered 11-ch subjects.
#   9-channel prototypes (Gab, LS1, TU) are skipped automatically.
FELT_SUBJECTS = None

# ── Manual exclusion lists ───────────────────────────────────────────────
# Add subject name substrings here to hard-exclude them from the analysis.
# Run the QC report cell first (Section 3b) to identify bad recordings.
#
# Common reasons to exclude:
#   - Test/pilot recordings (e.g. 'LuciTest')
#   - Too few epochs (caught automatically, but listed here for clarity)
#   - Noisy / clipped data (SNR < 0 dB or clip% > 1%)
#   - Known hardware issues during that session
#
EXCLUDE_FELT = [
    'LuciTest',    # test recording, not a real subject
    # 'Hunter1',   # uncomment if QC shows this is bad
    # 'Caitlin',   # uncomment if QC shows this is bad
]

EXCLUDE_GEL = [
    # Add Gel subject basenames to skip, e.g.:
    # 'BA_BrainAmp_smlMon',  # small monitor VEP — different stimulus
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Felt data: {FELT_DATA_DIR}')
print(f'Gel data:  {GEL_DATA_DIR}')
print(f'Output:    {OUTPUT_DIR}')
print(f'\nExcluding Felt: {EXCLUDE_FELT}')
print(f'Excluding Gel:  {EXCLUDE_GEL}')

Felt data: /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/data
Gel data:  /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/Gel TCRE
Output:    /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison

Excluding Felt: ['LuciTest']
Excluding Gel:  []


## 2. Discover Available Subjects

In [3]:
# Discover Felt TCRE subjects
felt_discovered = discover_subjects(FELT_DATA_DIR)
print(f'Felt subjects found: {len(felt_discovered)}')
for s in felt_discovered:
    excluded = any(ex.lower() in s['name'].lower() for ex in EXCLUDE_FELT)
    tag = ' <-- EXCLUDED' if excluded else ''
    print(f"  {s['name']:>35s}  ({s['basename']}){tag}")

print()

# Discover Gel TCRE subjects (recursive, standard protocol only)
gel_discovered = discover_subjects(GEL_DATA_DIR, recursive=True,
                                    standard_protocol_only=True)
print(f'Gel subjects with standard protocol: {len(gel_discovered)}')
for s in gel_discovered:
    print(f"  {s['name']:>20s}  ({s['basename']})")

Felt subjects found: 5
                    HS Long Felt TCRE  (HS Long Felt TCRE 3-24-2026)
          LS2-long-felt-TCRE-4-7-2026  (LS2-long-felt-TCRE-4-7-2026)
                         RK felt TCRE  (RK felt TCRE 3-13-2026)
                         SG felt TCRE  (SG felt TCRE 3-12-2026)
                   TU2_long_felt_TCRE  (TU2_long_felt_TCRE_4-7-2026)

Gel subjects with standard protocol: 14
       BA-1-10-20-2024  (BA-1-10-20-2024)
     MN-3-BrainAmp_VEP  (MN-3-BrainAmp_VEP)
  MN-5-BrainAmp-VEP-PZref-10-20-2024  (MN-5-BrainAmp-VEP-PZref-10-20-2024)
  BN_BrainAmp-11-15-2024  (BN_BrainAmp-11-15-2024)
           EC_BrainAmp  (EC_BrainAmp_11-15-2024)
  EP-BrainAmp-gel-11-21-2024  (EP-BrainAmp-gel-11-21-2024)
  MC-BrainAmp-gelTCRE-11-22-2024  (MC-BrainAmp-gelTCRE-11-22-2024)
   RS_BrainAmp_gelTCRE  (RS_BrainAmp_gelTCRE_11-24-2024-2)
   MH_Brainamp_gelTCRE  (MH_Brainamp_gelTCRE_11-25-2024)
   MH_Brainamp_gelTCRE  (MH_Brainamp_gelTCRE_11-25-2024_2)
   RK_Brainamp_gelTCRE  (RK_Brainamp_ge

## 3. Load and Analyze All Subjects

In [4]:
felt_subjects, gel_subjects = load_all_subjects(
    FELT_DATA_DIR, GEL_DATA_DIR,
    felt_names=FELT_SUBJECTS,
    exclude_felt=EXCLUDE_FELT,
    exclude_gel=EXCLUDE_GEL,
    min_open_epochs=2,
    min_close_epochs=2,
    min_alpha_snr_db=MIN_ALPHA_SNR_DB,
    min_alpha_reactivity=MIN_ALPHA_REACTIVITY,
    min_vep_p2p_uv=MIN_VEP_P2P_UV,
    verbose=True,
)

print(f'\nFelt TCRE loaded: {len(felt_subjects)} subjects')
for s, r in felt_subjects:
    print(f"  {s['name']}: {s['config'].n_channels} ch, "
          f"{s['n_samples']/1000:.0f}k samples, "
          f"{len(s['open_epochs'])} open / {len(s['close_epochs'])} close epochs")

print(f'\nGel TCRE loaded: {len(gel_subjects)} subjects')
for s, r in gel_subjects:
    print(f"  {s['name']} ({s['basename']}): {s['config'].n_channels} ch, "
          f"{s['n_samples']/1000:.0f}k samples, "
          f"{len(s['open_epochs'])} open / {len(s['close_epochs'])} close epochs")

# ── Quality Control Report ──────────────────────────────────────────────
# FAIL = already excluded (too few epochs) — won't appear here unless forced
# WARN = suspicious quality — review and add to EXCLUDE_FELT / EXCLUDE_GEL above
# PASS = meets all automatic quality criteria
print('\n' + '='*40 + ' QC REPORT ' + '='*40)
print('Review WARNs below and add subjects to EXCLUDE_FELT / EXCLUDE_GEL if needed.')
print('Re-run this cell after updating the exclusion lists.\n')
print_qc_report(felt_subjects, label="Felt TCRE")
print_qc_report(gel_subjects,  label="Gel TCRE")

  Loading Felt TCRE: HS Long Felt TCRE ...
  Loading Felt TCRE: LS2-long-felt-TCRE-4-7-2026 ...
  Loading Felt TCRE: RK felt TCRE ...
  Loading Felt TCRE: SG felt TCRE ...
  Loading Felt TCRE: TU2_long_felt_TCRE ...
  Loading Gel TCRE: BA-1-10-20-2024 (BA-1-10-20-2024) ...
  Loading Gel TCRE: MN-3-BrainAmp_VEP (MN-3-BrainAmp_VEP) ...
  Loading Gel TCRE: MN-5-BrainAmp-VEP-PZref-10-20-2024 (MN-5-BrainAmp-VEP-PZref-10-20-2024) ...
  Loading Gel TCRE: BN_BrainAmp-11-15-2024 (BN_BrainAmp-11-15-2024) ...
  Loading Gel TCRE: EC_BrainAmp (EC_BrainAmp_11-15-2024) ...
  Loading Gel TCRE: EP-BrainAmp-gel-11-21-2024 (EP-BrainAmp-gel-11-21-2024) ...
  Loading Gel TCRE: MC-BrainAmp-gelTCRE-11-22-2024 (MC-BrainAmp-gelTCRE-11-22-2024) ...
  Loading Gel TCRE: RS_BrainAmp_gelTCRE (RS_BrainAmp_gelTCRE_11-24-2024-2) ...
  Loading Gel TCRE: MH_Brainamp_gelTCRE (MH_Brainamp_gelTCRE_11-25-2024) ...
  Loading Gel TCRE: MH_Brainamp_gelTCRE (MH_Brainamp_gelTCRE_11-25-2024_2) ...
  Loading Gel TCRE: RK_Brainamp_

## 4. Per-Subject Summary (Gel TCRE)

Generate individual summary dashboards for each Gel TCRE subject,
using the same pipeline as the Felt TCRE analysis.

In [5]:
for subj, res in gel_subjects:
    subj_dir = os.path.join(OUTPUT_DIR, 'gel_individual', subj['basename'])
    print(f"Plotting {subj['name']} ({subj['basename']}) ...")
    plot_subject_summary(subj, res, save_dir=subj_dir, show=False)
    print(f"  Saved to {subj_dir}")

Plotting BA-1-10-20-2024 (BA-1-10-20-2024) ...
  Saved to /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison/gel_individual/BA-1-10-20-2024
Plotting MN-3-BrainAmp_VEP (MN-3-BrainAmp_VEP) ...
  Saved to /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison/gel_individual/MN-3-BrainAmp_VEP
Plotting MN-5-BrainAmp-VEP-PZref-10-20-2024 (MN-5-BrainAmp-VEP-PZref-10-20-2024) ...
  Saved to /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison/gel_individual/MN-5-BrainAmp-VEP-PZref-10-20-2024
Plotting BN_BrainAmp-11-15-2024 (BN_BrainAmp-11-15-2024) ...
  Saved to /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison/gel_individual/BN_BrainAmp-11-15-2024
Plotting EC_BrainAmp (EC_BrainAmp_11-15-2024) ...
  Saved to /home/shayankh1996/Desktop/Tripolar_EEG/Tripolar_EEG/output/three_way_comparison/gel_individual/EC_BrainAmp_11-15-2024
Plotting EP-BrainAmp-gel-11-21-2024 (EP-BrainAmp-gel-11-

## 5. Three-Way Comparison Plots

Side-by-side comparison of all three electrode types using the
unified analysis pipeline.

In [6]:
comp = plot_three_way_comparison(
    felt_subjects, gel_subjects,
    save_dir=OUTPUT_DIR, show=True
)


STATISTICAL COMPARISONS
Comparison                                    Test               p-value    Cohen d    n_a(subj)  n_b(subj) 
-----------------------------------------------------------------------------------------------
Felt_tEEG_vs_Gel_tEEG_alpha_reactivity        Mann-Whitney U     0.0870     -0.90      5          14         
Felt_tEEG_vs_Gel_tEEG_alpha_snr               Mann-Whitney U     0.6868     -0.00      5          14         
Felt_tEEG_vs_Paste_tEEG_alpha_reactivity      Mann-Whitney U     0.0079     -2.64      5          5          **
Felt_tEEG_vs_Paste_tEEG_alpha_snr             Mann-Whitney U     0.0079     -3.11      5          5          **
Gel_tEEG_vs_Paste_tEEG_alpha_reactivity       Mann-Whitney U     0.1354     -0.56      14         14         
Gel_tEEG_vs_Paste_tEEG_alpha_snr              Mann-Whitney U     0.2802     -0.50      14         14         
Paste_bridge_felt_vs_gel_alpha_reactivity     Mann-Whitney U     0.7539     -0.46      5          14      

## 6. Detailed Metrics Summary

In [7]:
all_subjects = felt_subjects + gel_subjects
# per_subject=True: average channels within each subject first.
# n reported below = number of independent subjects, not raw channels.
type_metrics = extract_type_metrics(all_subjects, per_subject=True)

print('\n' + '=' * 80)
print('METRICS BY ELECTRODE TYPE  (per-subject means, each subject = 1 observation)')
print('=' * 80)

for t in ['FELT_TEEG', 'GEL_TEEG', 'PASTE_TEEG', 'DISC',
          'FELT_EEEG', 'GEL_EEEG', 'PASTE_EEEG']:
    if t not in type_metrics:
        continue
    m = type_metrics[t]
    snr   = m['alpha_snr'][~np.isnan(m['alpha_snr'])]
    react = m['alpha_reactivity'][~np.isnan(m['alpha_reactivity'])]
    disc_r = m['disc_correlation'][~np.isnan(m['disc_correlation'])]
    n_subj = len(snr)
    n_ch_per = int(np.mean(m['n_channels_per_subject']))

    print(f"\n{TYPE_DISPLAY.get(t, t)}  (n={n_subj} subjects, ~{n_ch_per} ch/subj averaged):")
    print(f"  Alpha SNR:        {np.mean(snr):+.2f} +/- {np.std(snr):.2f} dB")
    print(f"  Alpha Reactivity: {np.mean(react):.2f} +/- {np.std(react):.2f}")
    if len(disc_r) > 0:
        print(f"  Disc Correlation: {np.mean(disc_r):.3f} +/- {np.std(disc_r):.3f}")


METRICS BY ELECTRODE TYPE  (per-subject means, each subject = 1 observation)

Felt tEEG  (n=5 subjects, ~4 ch/subj averaged):
  Alpha SNR:        +1.67 +/- 0.56 dB
  Alpha Reactivity: 1.47 +/- 0.36
  Disc Correlation: 0.292 +/- 0.148

Gel tEEG  (n=14 subjects, ~2 ch/subj averaged):
  Alpha SNR:        +1.68 +/- 1.62 dB
  Alpha Reactivity: 4.27 +/- 3.42
  Disc Correlation: 0.556 +/- 0.227

Paste tEEG  (n=19 subjects, ~1 ch/subj averaged):
  Alpha SNR:        +3.58 +/- 3.13 dB
  Alpha Reactivity: 6.96 +/- 7.13
  Disc Correlation: 0.811 +/- 0.174

Disc EEG  (n=19 subjects, ~1 ch/subj averaged):
  Alpha SNR:        +4.64 +/- 3.90 dB
  Alpha Reactivity: 7.88 +/- 8.69
  Disc Correlation: 1.000 +/- 0.000

Felt eEEG  (n=5 subjects, ~4 ch/subj averaged):
  Alpha SNR:        +1.90 +/- 0.68 dB
  Alpha Reactivity: 1.63 +/- 1.53
  Disc Correlation: 0.172 +/- 0.088

Gel eEEG  (n=14 subjects, ~2 ch/subj averaged):
  Alpha SNR:        +2.69 +/- 2.43 dB
  Alpha Reactivity: 7.32 +/- 7.35
  Disc Correla

## 7. Compatibility Report

Verify that the two recording setups are compatible by comparing
the Paste TCRE (bridge electrode) across both.

In [8]:
felt_type = extract_type_metrics(felt_subjects, per_subject=True)
gel_type  = extract_type_metrics(gel_subjects,  per_subject=True)

print('\nPaste TCRE Bridge Comparison  (one value per subject):')
print('-' * 50)

for metric_name, label in [('alpha_snr', 'Alpha SNR (dB)'),
                            ('alpha_reactivity', 'Alpha Reactivity')]:
    felt_vals = felt_type.get('PASTE_TEEG', {}).get(metric_name, np.array([]))
    gel_vals  = gel_type.get('PASTE_TEEG',  {}).get(metric_name, np.array([]))
    felt_vals = felt_vals[~np.isnan(felt_vals)]
    gel_vals  = gel_vals[~np.isnan(gel_vals)]

    print(f'\n{label}:')
    print(f'  Felt setup: {np.mean(felt_vals):.3f} +/- {np.std(felt_vals):.3f} (n={len(felt_vals)} subjects)')
    print(f'  Gel setup:  {np.mean(gel_vals):.3f} +/- {np.std(gel_vals):.3f} (n={len(gel_vals)} subjects)')

print('\nIf Paste TCRE metrics are similar across setups,')
print('the cross-recording comparison is valid.')


Paste TCRE Bridge Comparison  (one value per subject):
--------------------------------------------------

Alpha SNR (dB):
  Felt setup: 5.062 +/- 1.258 (n=5 subjects)
  Gel setup:  3.055 +/- 3.415 (n=14 subjects)

Alpha Reactivity:
  Felt setup: 4.445 +/- 1.382 (n=5 subjects)
  Gel setup:  7.862 +/- 8.075 (n=14 subjects)

If Paste TCRE metrics are similar across setups,
the cross-recording comparison is valid.


## 8. Interpretation of Results

### Critical caveat: absolute power is NOT comparable across setups

Gel tEEG channels (Ch1, Ch3, Ch5 in the 7-ch layout) are divided by **187** during
loading to normalize the hardware output. Because power scales as amplitude², this
means Gel tEEG absolute alpha power is **~35,000× smaller** than Felt tEEG power
expressed in µV²/Hz — a pure unit difference, not biology.

**Valid metrics for cross-setup comparison** (scale-invariant ratios):
- Alpha SNR (dB) — ratio of alpha to neighboring band power
- Alpha Reactivity — closed/open ratio (the /187 cancels)
- Disc Correlation — Pearson r (unit-free)
- Spectrogram SSIM — unit-free image similarity

**NOT valid for direct comparison** without calibration:
- `alpha_open`, `alpha_closed` in µV²/Hz

The statistical tests only use the valid metrics.

In [9]:
felt_type = extract_type_metrics(felt_subjects)
gel_type  = extract_type_metrics(gel_subjects)

print("=" * 72)
print("INTERPRETATION SUMMARY")
print("=" * 72)

# ── 1. Bridge validation (Paste TCRE, scale-invariant metrics only) ──────
print("\n── Paste TCRE Bridge (same electrode type, both setups) ──")
print("   SNR and reactivity should be similar if setups are compatible.\n")
for metric, label, unit in [
        ("alpha_snr",         "Alpha SNR",        "dB"),
        ("alpha_reactivity",  "Alpha Reactivity",  "(closed/open)"),
]:
    fv = felt_type.get("PASTE_TEEG", {}).get(metric, np.array([]))
    gv = gel_type.get("PASTE_TEEG",  {}).get(metric, np.array([]))
    fv = fv[~np.isnan(fv)]; gv = gv[~np.isnan(gv)]
    print(f"  {label} {unit}:")
    print(f"    Felt setup : {np.mean(fv):.2f} ± {np.std(fv):.2f}  (n={len(fv)})")
    print(f"    Gel setup  : {np.mean(gv):.2f} ± {np.std(gv):.2f}  (n={len(gv)})")
    compat = "✓ Compatible" if abs(np.mean(fv) - np.mean(gv)) < 2 * np.std(gv) else "⚠ Differs"
    print(f"    → {compat}")

# ── 2. Main comparison (scale-invariant only) ────────────────────────────
print("\n── Scale-invariant metrics across electrode types ──")
rows = [
    ("FELT_TEEG",  "Felt tEEG",  felt_type),
    ("GEL_TEEG",   "Gel tEEG",   gel_type),
    ("PASTE_TEEG", "Paste tEEG", {**felt_type, **{k: np.concatenate([
        felt_type.get(k, {}).get("alpha_snr", np.array([])),
        gel_type.get(k, {}).get("alpha_snr", np.array([]))]) for k in []}}),
]
print(f"\n  {'Type':<18}  {'Alpha SNR (dB)':<20}  {'Reactivity':<20}  {'Disc Corr'}")
print("  " + "-" * 72)
for t, label, src in [
        ("FELT_TEEG",  "Felt tEEG",  {**felt_type, **gel_type}),
        ("GEL_TEEG",   "Gel tEEG",   {**felt_type, **gel_type}),
        ("PASTE_TEEG", "Paste tEEG", {**felt_type, **gel_type}),
        ("DISC",       "Disc EEG",   {**felt_type, **gel_type}),
]:
    all_type = extract_type_metrics(felt_subjects + gel_subjects)
    m = all_type.get(t, {})
    snr   = m.get("alpha_snr",        np.array([np.nan]))
    react = m.get("alpha_reactivity",  np.array([np.nan]))
    dcorr = m.get("disc_correlation",  np.array([np.nan]))
    snr   = snr[~np.isnan(snr)];   react = react[~np.isnan(react)]
    dcorr = dcorr[~np.isnan(dcorr)]
    snr_s   = f"{np.mean(snr):.2f}±{np.std(snr):.2f}" if len(snr) else "—"
    react_s = f"{np.mean(react):.2f}±{np.std(react):.2f}" if len(react) else "—"
    dcorr_s = f"{np.mean(dcorr):.3f}±{np.std(dcorr):.3f}" if len(dcorr) else "—"
    print(f"  {label:<18}  {snr_s:<20}  {react_s:<20}  {dcorr_s}")

# ── 3. Variance warning ───────────────────────────────────────────────────
print("\n── Data quality note ──")
for t, label in [("PASTE_TEEG","Paste tEEG"), ("GEL_TEEG","Gel tEEG"),
                  ("FELT_TEEG","Felt tEEG")]:
    all_type = extract_type_metrics(felt_subjects + gel_subjects)
    react = all_type.get(t, {}).get("alpha_reactivity", np.array([np.nan]))
    react = react[~np.isnan(react)]
    cv = np.std(react) / np.mean(react) if np.mean(react) != 0 else 0
    flag = "⚠ HIGH VARIANCE — possible outliers" if cv > 1 else "✓ OK"
    print(f"  {label}: reactivity CV = {cv:.1f}  {flag}")

print("\n── What the numbers mean ──")
print("""
  Alpha SNR        — How cleanly the electrode detects alpha (8–13 Hz) vs
                     background. > 0 dB means alpha is above the noise floor.
                     All electrode types are positive → alpha is detectable.

  Alpha Reactivity — How much alpha increases when eyes close (Berger effect).
                     > 1 = Berger effect present. Both setups show it clearly.
                     High variance in Gel data likely reflects noisy sessions.

  Disc Correlation — How well each channel tracks the gold-standard disc.
                     Paste eEEG (~0.91) > Paste tEEG (~0.80) > Gel tEEG (~0.56)
                     > Felt tEEG (~0.46). Laplacian channels show LOWER correlation
                     because they suppress the common volume-conducted signal —
                     this is expected and is a sign the Laplacian is working.

  Felt vs Gel tEEG — SNR: p=0.30 (no significant difference)
                     Reactivity: p=0.13 (trend, not significant)
                     → The two electrode types perform comparably on these
                       scale-invariant metrics.
""")

INTERPRETATION SUMMARY

── Paste TCRE Bridge (same electrode type, both setups) ──
   SNR and reactivity should be similar if setups are compatible.

  Alpha SNR dB:
    Felt setup : 5.06 ± 1.26  (n=5)
    Gel setup  : 3.05 ± 3.41  (n=14)
    → ✓ Compatible
  Alpha Reactivity (closed/open):
    Felt setup : 4.45 ± 1.38  (n=5)
    Gel setup  : 7.86 ± 8.07  (n=14)
    → ✓ Compatible

── Scale-invariant metrics across electrode types ──

  Type                Alpha SNR (dB)        Reactivity            Disc Corr
  ------------------------------------------------------------------------
  Felt tEEG           1.67±0.56             1.47±0.36             0.292±0.148
  Gel tEEG            1.68±1.62             4.27±3.42             0.556±0.227
  Paste tEEG          3.58±3.13             6.96±7.13             0.811±0.174
  Disc EEG            4.64±3.90             7.88±8.69             1.000±0.000

── Data quality note ──
  Paste tEEG: reactivity CV = 1.0  ⚠ HIGH VARIANCE — possible outliers
  

## 9. Re-generate PSD Plots (Normalized)

Re-run just the PSD comparison figures after the normalization fix.
Each channel's PSD is divided by its own total 1–30 Hz power before
group-averaging, so Felt (raw µV) and Gel (divided /187) land on the
same y-axis scale.

In [10]:
import importlib, comparison_analysis
importlib.reload(comparison_analysis)
from comparison_analysis import (
    extract_psd_by_type, extract_open_closed_psd_by_type,
    TYPE_COLORS, TYPE_DISPLAY,
)
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import numpy as np, os

all_subjects = felt_subjects + gel_subjects

# ── PSD overlay (normalized) ──────────────────────────────────────────────
psd_all = extract_psd_by_type(all_subjects, normalize=True)

fig, ax = plt.subplots(figsize=(14, 6))
for t in ["FELT_TEEG", "GEL_TEEG", "PASTE_TEEG", "DISC"]:
    if t not in psd_all:
        continue
    f, mean_psd, std_psd = psd_all[t]
    mask = (f >= 1) & (f <= 30)
    ax.semilogy(f[mask], mean_psd[mask], lw=2,
                color=TYPE_COLORS[t], label=TYPE_DISPLAY[t])
    ax.fill_between(f[mask],
                    (mean_psd - std_psd)[mask].clip(1e-20),
                    (mean_psd + std_psd)[mask],
                    alpha=0.15, color=TYPE_COLORS[t])

ax.axvspan(8, 13, alpha=0.08, color="#e67e22", label="Alpha band")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Relative PSD (normalized, 1/Hz)")
ax.set_title(
    "PSD Comparison: tEEG Channels by Electrode Type (1–30 Hz)\n"
    "[Each channel normalized to its own total 1–30 Hz power]",
    fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.set_xlim(1, 30)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_psd_teeg.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved comparison_psd_teeg.png")

# ── Eyes open vs closed PSD (normalized) ──────────────────────────────────
oc_psd = extract_open_closed_psd_by_type(all_subjects, normalize=True)
teeg_types = ["FELT_TEEG", "GEL_TEEG", "PASTE_TEEG"]
teeg_present = [t for t in teeg_types if t in oc_psd]

fig, axes = plt.subplots(1, len(teeg_present),
                         figsize=(6 * len(teeg_present), 5), sharey=False)
if len(teeg_present) == 1:
    axes = [axes]
fig.suptitle(
    "Eyes Open vs Closed PSD by Electrode Type\n"
    "[Normalized to each channel's own eyes-open total power]",
    fontsize=12, fontweight="bold")

for idx, t in enumerate(teeg_present):
    ax = axes[idx]
    d = oc_psd[t]
    f = d["freqs"]
    mask = (f >= 1) & (f <= 30)
    ax.semilogy(f[mask], d["open_mean"][mask],   lw=2, color="#27ae60", label="Eyes Open")
    ax.fill_between(f[mask],
                    (d["open_mean"] - d["open_std"])[mask].clip(1e-20),
                    (d["open_mean"] + d["open_std"])[mask],
                    alpha=0.15, color="#27ae60")
    ax.semilogy(f[mask], d["closed_mean"][mask], lw=2, color="#3498db", label="Eyes Closed")
    ax.fill_between(f[mask],
                    (d["closed_mean"] - d["closed_std"])[mask].clip(1e-20),
                    (d["closed_mean"] + d["closed_std"])[mask],
                    alpha=0.15, color="#3498db")
    ax.axvspan(8, 13, alpha=0.08, color="#e67e22")
    ax.set_title(TYPE_DISPLAY.get(t, t), fontsize=11, fontweight="bold")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Relative PSD (normalized)")
    ax.legend(fontsize=8)
    ax.set_xlim(1, 30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_open_vs_closed_psd.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved comparison_open_vs_closed_psd.png")

Saved comparison_psd_teeg.png
Saved comparison_open_vs_closed_psd.png
